# Kaggle — Stanford Dogs ONLY | Forget-10 and Forget-15

**Two remaining deleted-class diagnostic experiments · Four-hour notebook-internal time budget · Kaggle T4 GPU**

This notebook is a **focused continuation of the previous evaluation design**, not a restart of all 12 experiments. It evaluates only **Stanford Dogs / Forget-10** and **Stanford Dogs / Forget-15**, calculating:

1. Student–reference Jensen–Shannon divergence on forgotten-class test images (nats).
2. Retrained-reference prediction entropy on those images (nats).
3. Teacher, reference and student **frozen-feature linear-probe accuracy** on forgotten classes (%).
4. Student-own and reference-own membership-attack AUC on **retained** train/test samples.

The default `public_demo` mode downloads **only** `jessicali9530/stanford-dogs-dataset` via Kaggle's ZIP endpoint, *not* kagglehub mounting; Internet must be enabled. It normalizes breed folder names and uses the same seeded 80/20 split algorithm and forgotten-class order as the prior public-demo notebook. If the Kaggle dataset contents have changed, these will be **new** runs; examine the dataset manifest before comparing numbers.

**Critical research qualification:** Public-demo results and reconstructed teacher/student/reference weights are NOT measurements on your original balanced datasets and historical original models. These figures can complete the *diagnostic table for the public-demo reconstruction*, but should not be represented as verified reproduction of your balanced-data manuscript results.

### Kaggle instructions

Open on Kaggle → **GPU T4 ×2** → enable **Internet** → Run All. The code uses one T4. The download/extraction takes place before the **240-minute internal training budget** starts. Model weights are checkpointed after **each epoch**, metrics are saved after **each completed scope**, and the last cell provides a backup archive to download or save in Kaggle outputs.

**Session persistence:** `/kaggle/working` may disappear after a session reset. Use Kaggle's **Save Version** and verify output artifacts exist, or download the backup archive. For the next session, attach saved outputs and configure `PREVIOUS_RESULTS_DIR` before running.

In [1]:
# 1. Imports and environment
import os, sys, json, time, random, re, hashlib, shutil, zipfile, tarfile, subprocess, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from IPython.display import display

assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle (kaggle.com/code).'
assert torch.cuda.is_available(), 'Select GPU in Kaggle Settings -> Accelerator.'
DEVICE=torch.device('cuda:0')
print('Torch', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
print('Total GPUs:',torch.cuda.device_count(), '(this notebook deliberately uses only GPU 0)')

Torch 2.11.0+cu128 GPU: Tesla T4
Total GPUs: 2 (this notebook deliberately uses only GPU 0)


## 1 · Configuration: Stanford Dogs, Forget-10 and Forget-15 only

The training settings and probe/MIA definitions are preserved from your previous missing-metrics evaluation. A fresh reconstruction needs one teacher plus two students and two reference models unless compatible checkpoints are restored. The four-hour budget is *only an internal stop rule*, not an extension of Kaggle GPU access.

In [2]:
# ================== USER CONFIGURATION ==================
DATA_MODE = 'public_demo'  # diagnostics only; NOT the original balanced dataset

# For original balanced archives/folders, place originals in /kaggle/input
# via Add Input if using your original datasets in a future study; default is public_demo.
ORIGINAL_INPUT_OVERRIDES = {
    # 'Caltech-101': '/kaggle/input/my-balanced-data/caltech balanced dataset1.zip',
    # 'Caltech-256': '/kaggle/input/my-balanced-data/Caltech 256 balanced dataset.zip',
    # 'CIFAR-100': '/kaggle/input/my-balanced-data/cifar100.zip',
    # 'Stanford Dogs': '/kaggle/input/my-balanced-data/Standford dog balanced dataset.zip',
}

# Optional pre-trained teacher, original student, and aligned-reference checkpoints.
# Keys: ('Dataset', 0, 'teacher') or ('Dataset', 5/10/15, 'student'/'reference').
EXTERNAL_CHECKPOINTS = {
    # ('Caltech-101', 5, 'student'): '/kaggle/input/my-models/cal101_forget5_student.pth',
}
ALLOW_UNVERIFIED_EXTERNAL = False  # do not set True for publication claims
AUTO_TRAIN_MISSING_MODELS = True   # train original-like teacher/student + matched references if absent

# Resume: attach the OUTPUT of your previous Kaggle notebook version as an input.
PREVIOUS_BACKUP_ZIP = None  # e.g. /kaggle/input/saved-backup/stanford_dogs_10_15_backup.zip
PREVIOUS_RESULTS_DIR = None  # e.g. '/kaggle/input/previous-evaluation/deleted_class_diagnostics'

# 240 min = four-hour internal max, measured after dataset preparation.
TIME_BUDGET_MINUTES = 240
RUN_DATASETS = ['Stanford Dogs']
FORGET_SCOPES = [10, 15]
MAX_NEW_EXPERIMENTS_PER_SESSION = None  # optional 1 or 2 for controlled sessions

IMAGE_SIZE = 224
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 2
TRAIN_SEED = 42
TEACHER_EPOCHS = 30
STUDENT_EPOCHS = 30
REFERENCE_EPOCHS = 30
TEACHER_LR = 1e-4
STUDENT_LR = 3e-4
REFERENCE_LR = 1e-4
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.7
USE_AMP = True

# Frozen-feature probing uses ALL forgotten train/test samples by default.
# MIA per-class caps keep inference manageable on large datasets.
MIA_PAIRS_PER_RETAINED_CLASS = 12
MIA_ATTACK_TEST_FRACTION = 0.30
CACHE_MODEL_FEATURES = True

# CoAtNet defaults are usually ImageNet normalization; verify against the
# exact timm.resolve_model_data_config used in original model runs.
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)

OUTPUT_DIR = Path('/kaggle/working/deleted_class_diagnostics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path('/kaggle/working/deleted_class_dataset_cache')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
# Start the four-hour training clock in the RUN cell, AFTER dataset downloads.
START_MONOTONIC = None
DEADLINE = None

if PREVIOUS_BACKUP_ZIP:
    backup=Path(PREVIOUS_BACKUP_ZIP)
    if not backup.is_file(): raise FileNotFoundError(f'Previous backup ZIP not found: {backup}')
    target=DATA_ROOT/'restored_backup'
    target.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(backup) as z:
        for member in z.infolist():
            if not (target/member.filename).resolve().is_relative_to(target.resolve()):
                raise ValueError('Unsafe backup ZIP entry: '+member.filename)
        z.extractall(target)
    PREVIOUS_RESULTS_DIR=str(target/'deleted_class_diagnostics')
    print('Unpacked previous checkpoint/results ZIP:',backup)
if PREVIOUS_RESULTS_DIR:
    previous=Path(PREVIOUS_RESULTS_DIR)
    if not previous.is_dir():
        raise FileNotFoundError(f'Previous results not attached: {previous}')
    if previous.resolve() == OUTPUT_DIR.resolve():
        raise ValueError('PREVIOUS_RESULTS_DIR must be a separate saved-version folder.')
    shutil.copytree(previous, OUTPUT_DIR, dirs_exist_ok=True)
    print('Restored saved models and experiment metrics from:', previous)

print('DATA_MODE:', DATA_MODE, '| time budget:', TIME_BUDGET_MINUTES, 'minutes')
print('Output:', OUTPUT_DIR)

assert RUN_DATASETS == ['Stanford Dogs'] and FORGET_SCOPES == [10, 15], 'Only Stanford Dogs Forget-10/15 may run.'

DATA_MODE: public_demo | time budget: 240 minutes
Output: /kaggle/working/deleted_class_diagnostics


## 2 · Download **Stanford Dogs only** (no Kagglehub mount)

This cell uses the same direct-ZIP, breed-name normalization, original-split-if-provided or otherwise seeded 80/20 split logic as your prior public-demo run. It never fetches Caltech or CIFAR. If Kaggle Internet is off, it reports that problem explicitly instead of starting training.

In [3]:
from pathlib import Path
import os, zipfile, requests, socket

assert DATA_MODE == 'public_demo'
assert RUN_DATASETS == ['Stanford Dogs']

DOG_KAGGLE_ID = 'jessicali9530/stanford-dogs-dataset'
DOG_SOURCE_URL = 'https://www.kaggle.com/api/v1/datasets/download/' + DOG_KAGGLE_ID
DOG_ZIP = DATA_ROOT / 'Stanford-Dogs_public_download.zip'
DOG_EXTRACTED = DATA_ROOT / 'Stanford-Dogs_public_extracted'
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DATASET_FOLDERS = {'Stanford Dogs': 'Stanford-Dogs'}
EXPECTED_ORIGINAL = {'Stanford Dogs': (120, 8400, 3600)}
ORIGINAL_FORGET_15 = {'Stanford Dogs': [
    'borzoi','Brittany_spaniel','American_Staffordshire_terrier',
    'giant_schnauzer','Irish_water_spaniel','Greater_Swiss_Mountain_dog',
    'Gordon_setter','Chihuahua','Brabancon_griffon','chow',
    'toy_poodle','Walker_hound','Boston_bull','basenji','Rhodesian_ridgeback'
]}

# Optional local ZIP path / attached files: leave None for the Kaggle API download.
LOCAL_DOG_ARCHIVE = None


def safe_extract_zip(src, destination):
    src = Path(src)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(src) as archive:
        for info in archive.infolist():
            final = (root / info.filename).resolve()
            if not final.is_relative_to(root):
                raise ValueError('Unsafe path in ZIP: ' + info.filename)
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe symlink in ZIP: ' + info.filename)
        archive.extractall(root)


def ensure_public_dog_zip():
    if LOCAL_DOG_ARCHIVE:
        path = Path(LOCAL_DOG_ARCHIVE)
        if not zipfile.is_zipfile(path):
            raise FileNotFoundError(f'Local DOG ZIP not found/invalid: {path}')
        print('Using supplied Stanford Dogs archive:', path)
        return path
    if DOG_ZIP.is_file() and zipfile.is_zipfile(DOG_ZIP):
        print('Using cached Stanford Dogs ZIP:', DOG_ZIP)
        return DOG_ZIP
    print('Downloading Stanford Dogs directly as ZIP (no kagglehub mount)...', flush=True)
    # Avoid retries on DNS failure; explicit guidance to enable Internet in Kaggle Settings.
    try:
        socket.getaddrinfo('www.kaggle.com', 443)
    except OSError as e:
        raise RuntimeError('Kaggle Internet is disabled or DNS is unavailable. '
                           'Enable Settings > Internet and rerun this cell.') from e
    temp = DOG_ZIP.with_name(DOG_ZIP.name + '.part')
    temp.unlink(missing_ok=True)
    received = 0
    try:
        with requests.get(DOG_SOURCE_URL, stream=True, timeout=(30, 240)) as response:
            response.raise_for_status()
            with temp.open('wb') as fh:
                for chunk in response.iter_content(chunk_size=2 * 1024 * 1024):
                    if chunk:
                        fh.write(chunk)
                        received += len(chunk)
                        if received // (128 * 1024 * 1024) > (received-len(chunk)) // (128*1024*1024):
                            print(f'  received {received / (1024*1024):.0f} MiB', flush=True)
        if not zipfile.is_zipfile(temp):
            raise ValueError('Kaggle did not return a valid ZIP. Check dataset and network access.')
        os.replace(temp, DOG_ZIP)
    except Exception:
        temp.unlink(missing_ok=True)
        raise
    print(f'Downloaded {received/(1024*1024):.1f} MiB', flush=True)
    return DOG_ZIP


def directories_under(base, max_depth=7):
    base = Path(base)
    for current, dirs, _ in os.walk(base):
        path = Path(current)
        if len(path.relative_to(base).parts) > max_depth:
            dirs[:] = []
            continue
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        yield path


def image_class_count(folder):
    if not Path(folder).is_dir():
        return 0
    return sum(1 for child in Path(folder).iterdir()
               if child.is_dir() and any(f.is_file() and f.suffix.lower() in IMAGE_EXT
                                         for f in child.iterdir()))


def find_child(folder, options):
    return next((p for p in Path(folder).iterdir()
                 if p.is_dir() and p.name.lower() in options), None)


def normalize_dog_class(folder_name):
    # Same WordNet-normalization logic as the previously run public_demo cell.
    if len(folder_name) > 10 and folder_name.startswith('n') and folder_name[1:9].isdigit() and folder_name[9] == '-':
        return folder_name[10:]
    return folder_name


def link_classes(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for cls in Path(source).iterdir():
        if not cls.is_dir():
            continue
        target = destination / normalize_dog_class(cls.name)
        if target.is_symlink() and target.resolve() != cls.resolve():
            target.unlink()
        elif target.exists() and not target.is_symlink():
            raise RuntimeError(f'Existing conflicting class folder: {target}')
        if not target.exists():
            target.symlink_to(cls.resolve(), target_is_directory=True)
    return destination


def split_dog_dataset(base):
    roots = list(directories_under(base))
    # Preserve an existing train/test arrangement where provided.
    for root in roots:
        train = find_child(root, {'train', 'training'})
        test = find_child(root, {'test', 'testing'})
        if train is not None and test is not None and image_class_count(train) >= 2 and image_class_count(test) >= 2:
            stage = DATA_ROOT / 'Stanford-Dogs_demo_normalized'
            return link_classes(train, stage/'train'), link_classes(test, stage/'test')

    # Match original public_demo 80/20 seeded split, without image copying.
    best = max(roots, key=image_class_count)
    if image_class_count(best) < 2:
        raise RuntimeError(f'Stanford Dogs image class folders not found in {base}')
    stage = DATA_ROOT / 'Stanford-Dogs_demo_split'
    mark = stage / '.done'
    if not mark.exists():
        rng = np.random.default_rng(TRAIN_SEED)
        classes = sorted((p for p in best.iterdir() if p.is_dir()), key=lambda p: p.name)
        for cls in classes:
            images = sorted(f for f in cls.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXT)
            if len(images) < 2:
                continue
            label = normalize_dog_class(cls.name)
            count_test = max(1, round(len(images) * .20))
            test_indices = set(rng.permutation(len(images))[:count_test].tolist())
            for idx, src in enumerate(images):
                split = 'test' if idx in test_indices else 'train'
                folder = stage / split / label
                folder.mkdir(parents=True, exist_ok=True)
                link = folder / src.name
                if not link.exists():
                    link.symlink_to(src.resolve())
        mark.touch()
    return stage/'train', stage/'test'


archive = ensure_public_dog_zip()
marker = DOG_EXTRACTED/'.complete'
if not marker.is_file():
    print('Extracting Stanford Dogs archive...', flush=True)
    safe_extract_zip(archive, DOG_EXTRACTED)
    marker.touch()
else:
    print('Using extracted Stanford Dogs dataset.')

try:
    train_dir, test_dir = split_dog_dataset(DOG_EXTRACTED)
except RuntimeError:
    # Some Kaggle packages contain a nested ZIP.
    inner_zips = sorted(f for f in DOG_EXTRACTED.rglob('*.zip') if f.is_file())
    if not inner_zips:
        raise
    for i, inner in enumerate(inner_zips):
        dst = DOG_EXTRACTED / f'_nested_{i}'
        nested_marker = dst/'.complete'
        if not nested_marker.is_file():
            print('Extracting nested ZIP:', inner.name, flush=True)
            safe_extract_zip(inner, dst)
            nested_marker.touch()
    train_dir, test_dir = split_dog_dataset(DOG_EXTRACTED)

DATASET_PATHS = {'Stanford Dogs': (train_dir, test_dir)}
print('\nSTANFORD DOGS READY')
print('Train:', train_dir)
print('Test: ', test_dir)
print('Mode: ', DATA_MODE)
print('Only scopes:', FORGET_SCOPES)
assert RUN_DATASETS == ['Stanford Dogs']

  received 128 MiB
  received 256 MiB
  received 384 MiB
  received 512 MiB
  received 640 MiB
Downloaded 750.5 MiB
Extracting Stanford Dogs archive...

STANFORD DOGS READY
Train: /kaggle/working/deleted_class_dataset_cache/Stanford-Dogs_demo_split/train
Test:  /kaggle/working/deleted_class_dataset_cache/Stanford-Dogs_demo_split/test
Mode:  public_demo
Only scopes: [10, 15]


## 3 · Verify category identities, dataset split and forgotten breeds

The original forgotten-breed order is preserved. Only Forget-10 (first ten breeds) and Forget-15 (all fifteen breeds) are evaluated. The dataset fingerprint is saved and used to reject incompatible checkpoints; for comparisons with earlier diagnostic rows, ensure the source dataset version and split are unchanged.

In [5]:
class RawCIFAR100(Dataset):
    # Optional fallback for public-demo CIFAR-100 binary distributions.
    def __init__(self, root, train=True):
        self.base=datasets.CIFAR100(str(root),train=train,download=False)
        original=self.base.classes
        self.classes=sorted(original)
        mapping={name:i for i,name in enumerate(self.classes)}
        self.targets=[mapping[original[y]] for y in self.base.targets]
    def __len__(self): return len(self.base)
    def __getitem__(self,i):
        image,original_label=self.base[i]
        return image,self.targets[i]

RAW_DATA={}
DATASET_MANIFEST={}
FORGET_CLASSES={}
for dataset_name in RUN_DATASETS:
    tr,te=DATASET_PATHS[dataset_name]
    if dataset_name=='CIFAR-100' and (tr/'cifar-100-python'/'train').is_file():
        train_data=RawCIFAR100(tr,train=True)
        test_data=RawCIFAR100(te,train=False)
    else:
        train_data=datasets.ImageFolder(str(tr),transform=None)
        test_data=datasets.ImageFolder(str(te),transform=None)
    class_names=train_data.classes
    if class_names!=test_data.classes:
        raise ValueError(f'{dataset_name} train/test class indexes differ! Stop: cannot safely evaluate.')
    expected=EXPECTED_ORIGINAL[dataset_name]
    actual=(len(class_names),len(train_data),len(test_data))
    if DATA_MODE=='original_balanced' and actual!=expected:
        raise ValueError(f'{dataset_name}: original data should be {expected} (classes,train,test), found {actual}. '
                         'Resolve the balanced dataset/split mismatch before publication.')
    forgotten=ORIGINAL_FORGET_15[dataset_name]
    unknown=[x for x in forgotten if x not in class_names]
    if unknown:
        raise ValueError(f'{dataset_name}: original forgotten categories missing: {unknown}. '
                         'Check dataset version and names; do NOT substitute arbitrary indices.')
    # Manifest independent of the temporary Kaggle extraction root.
    h=hashlib.sha256()
    for split,ds,root in [('train',train_data,tr),('test',test_data,te)]:
        if isinstance(ds,RawCIFAR100):
            h.update(split.encode()); h.update(ds.base.data.tobytes());
            h.update(np.asarray(ds.targets,dtype=np.int32).tobytes())
        else:
            for file_path,label in ds.samples:
                p=Path(file_path)
                h.update(f'{split}|{p.relative_to(root)}|{label}|{p.stat().st_size}\n'.encode())
    fingerprint=h.hexdigest()[:20]
    RAW_DATA[dataset_name]={'train':train_data,'test':test_data,'classes':class_names}
    DATASET_MANIFEST[dataset_name]={'fingerprint':fingerprint,'classes':class_names,
                                     'counts':{'train':len(train_data),'test':len(test_data)},
                                     'paths':[str(tr),str(te)],'data_mode':DATA_MODE}
    for n in FORGET_SCOPES:
        FORGET_CLASSES[(dataset_name,n)]=[class_names.index(cls) for cls in forgotten[:n]]
    print(f'{dataset_name:16} {len(class_names)} classes; {len(train_data)} train; '
          f'{len(test_data)} test | forget-5/10/15: exact names found | id={fingerprint}')

(DATA_ROOT/'manifest.json').write_text(json.dumps(DATASET_MANIFEST,indent=2))
(OUTPUT_DIR/'dataset_manifest.json').write_text(json.dumps(DATASET_MANIFEST,indent=2))
print('Dataset identity and forgotten classes verified.')

Stanford Dogs    120 classes; 16469 train; 4111 test | forget-5/10/15: exact names found | id=e14fad367e652c8fd317
Dataset identity and forgotten classes verified.


## 4 · Model protocol and strict checkpoint identity

Retains the previous ResNet-18 training choices, normalization, and original reconstruction hyperparameters. If you have checkpoints from the interrupted run, attach those saved files and set `PREVIOUS_RESULTS_DIR` in Configuration; otherwise the necessary teacher/student/reference models will be retrained from scratch.

In [6]:
class TimeBudgetPause(Exception): pass

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=True

def transform_for(training=False):
    operations=[transforms.Resize((IMAGE_SIZE,IMAGE_SIZE))]
    if training:
        operations.append(transforms.RandomHorizontalFlip(p=0.5))
    operations += [transforms.ToTensor(),transforms.Normalize(NORMALIZE_MEAN,NORMALIZE_STD)]
    return transforms.Compose(operations)

class View(Dataset):
    def __init__(self,raw,indices,augment=False):
        self.raw=raw; self.indices=np.asarray(indices,dtype=np.int64)
        self.transform=transform_for(augment)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        image,label=self.raw[int(self.indices[i])]
        return self.transform(image.convert('RGB')),int(label)

def get_y(ds):
    return np.asarray(ds.targets,dtype=np.int64)

def loader_view(raw,indices,training=False,batch_size=None,seed=42):
    ds=View(raw,indices,augment=training)
    return DataLoader(ds,batch_size=batch_size or (BATCH_SIZE if training else EVAL_BATCH_SIZE),
                      shuffle=training,num_workers=NUM_WORKERS,pin_memory=True,
                      generator=torch.Generator().manual_seed(seed),persistent_workers=False)

def make_resnet(num_classes, pretrained=False):
    weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model=models.resnet18(weights=weights)
    model.fc=nn.Linear(model.fc.in_features,num_classes)
    return model

def model_slug(dataset_name,scope,role):
    return dataset_name.lower().replace(' ','_').replace('-','_')+f'_forget{scope}_{role}'

def model_metadata(dataset_name,scope,role):
    dataset=DATASET_MANIFEST[dataset_name]
    forgotten=[] if role=='teacher' else [dataset['classes'][i] for i in FORGET_CLASSES[(dataset_name,scope)]]
    params={'role':role,'scope':scope,'dataset':dataset_name,
            'dataset_fingerprint':dataset['fingerprint'],'classes':dataset['classes'],
            'forgotten':forgotten,'seed':TRAIN_SEED,'size':IMAGE_SIZE,
            'normalize_mean':list(NORMALIZE_MEAN),'normalize_std':list(NORMALIZE_STD),
            'initialization':'random' if role=='student' else 'imagenet_resnet18_v1',
            'epochs':{'teacher':TEACHER_EPOCHS,'student':STUDENT_EPOCHS,'reference':REFERENCE_EPOCHS}[role],
            'lr':{'teacher':TEACHER_LR,'student':STUDENT_LR,'reference':REFERENCE_LR}[role],
            'batch_size':BATCH_SIZE,'amp':USE_AMP,
            'protocol':'deleted_diagnostics_v2',
            'student_alpha':KD_ALPHA if role=='student' else None,
            'student_temperature':KD_TEMPERATURE if role=='student' else None}
    params['experiment_id']=hashlib.sha256(json.dumps(params,sort_keys=True).encode()).hexdigest()[:20]
    return params

def paths_for(meta):
    base=OUTPUT_DIR/'checkpoints'
    base.mkdir(exist_ok=True)
    stem=model_slug(meta['dataset'],meta['scope'],meta['role'])+'_'+meta['experiment_id']
    return base/(stem+'.pt'),base/(stem+'.progress.pt')

def load_checkpoint(path,meta,allow_unverified=False):
    payload=torch.load(str(path),map_location='cpu',weights_only=True)
    if isinstance(payload,dict) and 'state_dict' in payload:
        state=payload['state_dict'];existing=payload.get('metadata',{})
    elif isinstance(payload,dict) and 'model_state_dict' in payload:
        state=payload['model_state_dict'];existing=payload.get('metadata',{})
    elif isinstance(payload,dict) and all(isinstance(v,torch.Tensor) for v in payload.values()):
        state=payload;existing={}
    else: raise ValueError(f'Unknown checkpoint format: {path}')
    if existing.get('experiment_id')!=meta['experiment_id']:
        if not allow_unverified:
            raise ValueError(f'Checkpoint {path} has missing/mismatched manifest ID. '
                             'Do not compare mismatched models. Use a verified checkpoint or explicitly opt into unverified import.')
        warnings.warn('Loading unverified EXTERNAL weights: do not describe as protocol-matched.')
    clean={k.removeprefix('module.'):v for k,v in state.items()}
    model=make_resnet(len(meta['classes']),pretrained=False)
    model.load_state_dict(clean,strict=True)
    if existing.get('experiment_id') != meta['experiment_id']:
        provenance='unverified_external'
    elif payload.get('origin')=='reconstructed':
        provenance='reconstructed'
    elif payload.get('origin')=='original_saved':
        provenance='verified_original'
    else:
        provenance='verified_identity_unknown_origin'
    return model.to(DEVICE),provenance

def output_checkpoint(dataset_name,scope,role):
    meta=model_metadata(dataset_name,scope,role)
    final,progress=paths_for(meta)
    if final.is_file():
        model,provenance=load_checkpoint(final,meta)
        return model,meta,final,provenance
    key=(dataset_name,scope,role)
    if key in EXTERNAL_CHECKPOINTS:
        path=Path(EXTERNAL_CHECKPOINTS[key])
        if not path.is_file(): raise FileNotFoundError(path)
        model,provenance=load_checkpoint(path,meta,allow_unverified=ALLOW_UNVERIFIED_EXTERNAL)
        return model,meta,path,provenance
    if not AUTO_TRAIN_MISSING_MODELS:
        raise FileNotFoundError(f'Missing {key}. Supply checkpoint in EXTERNAL_CHECKPOINTS or allow training.')
    model=train_missing_model(dataset_name,scope,role,meta,final,progress)
    return model,meta,final,'reconstructed'

print('Model metadata and strict checkpoint validation ready.')

Model metadata and strict checkpoint validation ready.


## 5 · Train missing models — checkpoints after every epoch

A newly reconstructed student/reference is **not** a recovered historical student or reference. The reference uses retained training samples only. The checkpoint names include a dataset fingerprint and model configuration hash.

In [7]:
def train_missing_model(name,scope,role,meta,final,progress):
    seed=int(TRAIN_SEED)+(scope*101 if role!='teacher' else 0)+{'teacher':0,'student':7,'reference':13}[role]
    seed_all(seed)
    raw=RAW_DATA[name]['train']
    labels=get_y(raw)
    removed=set(FORGET_CLASSES[(name,scope)]) if role!='teacher' else set()
    indices=np.where(~np.isin(labels,list(removed)))[0]
    if len(indices)==0: raise ValueError('No retained training samples!')
    train_loader=loader_view(raw,indices,training=True,seed=seed)

    # Use an ImageNet starting backbone for full teacher and retrained reference.
    model=make_resnet(len(meta['classes']),pretrained=(role!='student')).to(DEVICE)
    teacher=None
    if role=='student':
        teacher,tmeta,tfile,tprov=output_checkpoint(name,0,'teacher')
        teacher.eval()
        for p in teacher.parameters():p.requires_grad_(False)
    lr={'teacher':TEACHER_LR,'student':STUDENT_LR,'reference':REFERENCE_LR}[role]
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    amp=bool(USE_AMP and DEVICE.type=='cuda')
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1
    history=[]
    if progress.exists():
        previous=torch.load(progress,map_location='cpu',weights_only=True)
        if previous.get('experiment_id')!=meta['experiment_id']:
            raise ValueError(f'Progress checkpoint mismatch: {progress}')
        model.load_state_dict(previous['model_state_dict'])
        opt.load_state_dict(previous['optimizer_state_dict'])
        scaler.load_state_dict(previous['scaler_state_dict'])
        history=previous.get('history',[])
        start_epoch=int(previous['epoch'])+1
        print('Resuming',name,role,scope,'at epoch',start_epoch,flush=True)
    epochs=meta['epochs']
    print(f'BEGIN {name} | {role} | Forget-{scope} | N-train={len(indices)} '
          f'| epochs {start_epoch}-{epochs}',flush=True)
    for epoch in range(start_epoch,epochs+1):
        model.train()
        t0=time.monotonic(); total=0;loss_total=0.
        for xb,yb in train_loader:
            xb=xb.to(DEVICE,non_blocking=True);yb=yb.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',enabled=amp):
                logits=model(xb)
                if role=='student':
                    with torch.no_grad():
                        target=teacher(xb)
                    soft=F.kl_div(F.log_softmax(logits/KD_TEMPERATURE,dim=1),
                                  F.softmax(target/KD_TEMPERATURE,dim=1),
                                  reduction='batchmean')*(KD_TEMPERATURE**2)
                    loss=KD_ALPHA*soft+(1-KD_ALPHA)*F.cross_entropy(logits,yb)
                else:
                    loss=F.cross_entropy(logits,yb)
            scaler.scale(loss).backward()
            scaler.step(opt);scaler.update()
            n=len(yb);total+=n;loss_total+=float(loss.detach().cpu())*n
        record={'epoch':epoch,'loss':round(loss_total/max(total,1),6),
                'epoch_seconds':round(time.monotonic()-t0,1)}
        history.append(record)
        payload={'experiment_id':meta['experiment_id'],'epoch':epoch,
                 'model_state_dict':model.state_dict(),
                 'optimizer_state_dict':opt.state_dict(),
                 'scaler_state_dict':scaler.state_dict(),'history':history}
        tmp=progress.with_name(progress.name+'.tmp')
        torch.save(payload,tmp); os.replace(tmp,progress)
        print(f"  epoch {epoch:2}/{epochs}: loss={record['loss']:.4f}, "
              f"time={record['epoch_seconds']:.1f}s",flush=True)
        if DEADLINE is not None and time.monotonic()>=DEADLINE and epoch<epochs:
            raise TimeBudgetPause(f'{name} {role} Forget-{scope} safely paused after epoch '
                                  f'{epoch}/{epochs}; checkpoint saved.')
    result={'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'metadata':meta,'origin':'reconstructed','history':history}
    tmp=final.with_name(final.name+'.tmp')
    torch.save(result,tmp);os.replace(tmp,final)
    progress.unlink(missing_ok=True)
    pd.DataFrame(history).to_csv(final.with_suffix('.history.csv'),index=False)
    if teacher is not None: del teacher
    print('MODEL SAVED:',final,flush=True)
    return model

print('Epoch-resumable teacher/student/reference training functions ready.')

Epoch-resumable teacher/student/reference training functions ready.


## 6 · Deleted-class JSD, entropy and probing; retained-set attack AUC

Exactly the previous definitions: paired student/reference predictions on forgotten test data, frozen-feature probes fit on forgotten training examples, and independently trained retained-member MIA classifiers for student and reference.

In [8]:
def select_indices(name,scope):
    train_y=get_y(RAW_DATA[name]['train'])
    test_y=get_y(RAW_DATA[name]['test'])
    classes=np.asarray(FORGET_CLASSES[(name,scope)],dtype=np.int64)
    return {
      'forget_train':np.flatnonzero(np.isin(train_y,classes)),
      'forget_test':np.flatnonzero(np.isin(test_y,classes)),
      'retain_train':np.flatnonzero(~np.isin(train_y,classes)),
      'retain_test':np.flatnonzero(~np.isin(test_y,classes)),
    }

def mia_pairs(name,scope):
    # The SAME per-class sampling is used to evaluate each role.
    ytr=get_y(RAW_DATA[name]['train']); yte=get_y(RAW_DATA[name]['test'])
    forgotten=set(FORGET_CLASSES[(name,scope)])
    rng=np.random.default_rng(TRAIN_SEED+scope*37)
    members=[];nonmembers=[]
    for c in range(len(RAW_DATA[name]['classes'])):
        if c in forgotten:continue
        tr=np.flatnonzero(ytr==c);te=np.flatnonzero(yte==c)
        n=min(len(tr),len(te),MIA_PAIRS_PER_RETAINED_CLASS)
        if n<4: continue
        members.extend(rng.choice(tr,n,replace=False).tolist())
        nonmembers.extend(rng.choice(te,n,replace=False).tolist())
    if len(members)<20:raise ValueError(f'Insufficient retained members/nonmembers: {name} {scope}')
    return np.asarray(members,dtype=np.int64),np.asarray(nonmembers,dtype=np.int64)

@torch.inference_mode()
def compute_outputs(model,raw,indices):
    hook_features=[]
    def before_fc(mod,inp):hook_features.append(inp[0].detach().cpu().numpy())
    handle=model.fc.register_forward_pre_hook(before_fc)
    p_list=[];y_list=[];f_list=[]
    try:
        model.eval()
        loader=loader_view(raw,indices,training=False,seed=TRAIN_SEED)
        for xb,yb in loader:
            xb=xb.to(DEVICE,non_blocking=True)
            hook_features.clear()
            logits=model(xb)
            probs=F.softmax(logits,dim=1).float().cpu().numpy()
            if len(hook_features)!=1:raise AssertionError('ResNet-18 feature hook failed.')
            p_list.append(probs)
            f_list.append(hook_features[0])
            y_list.append(yb.numpy())
    finally:handle.remove()
    return {'probs':np.concatenate(p_list),
            'features':np.concatenate(f_list),
            'labels':np.concatenate(y_list)}

def cached_outputs(name,scope,role,model,meta,subset,indices):
    digest=hashlib.sha256(indices.astype('<i8').tobytes()).hexdigest()[:16]
    filename=f'{model_slug(name,scope,role)}_{meta["experiment_id"]}_{subset}_{digest}.npz'
    cache_dir=OUTPUT_DIR/'feature_caches';cache_dir.mkdir(exist_ok=True)
    cache=cache_dir/filename
    if CACHE_MODEL_FEATURES and cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            return {key:z[key] for key in ('probs','features','labels')}
    split='train' if subset.endswith('_train') else 'test'
    answer=compute_outputs(model,RAW_DATA[name][split],indices)
    if CACHE_MODEL_FEATURES:
        temp=cache.with_name(cache.name+'.tmp')
        with temp.open('wb') as fh:
            np.savez(fh,**answer)
        os.replace(temp,cache)
    return answer

def mean_jsd(p,q):
    if p.shape!=q.shape:raise ValueError(f'Paired probabilities differ: {p.shape} vs {q.shape}')
    eps=1e-12
    p=np.clip(p.astype('float64'),eps,1.);q=np.clip(q.astype('float64'),eps,1.)
    m=.5*(p+q)
    per=.5*np.sum(p*np.log(p/m),axis=1)+.5*np.sum(q*np.log(q/m),axis=1)
    return float(np.mean(per))

def entropy_nats(p):
    p=np.clip(p.astype('float64'),1e-12,1.)
    return float(np.mean(-np.sum(p*np.log(p),axis=1)))

def probe_accuracy(train_outputs,test_outputs):
    ytr=train_outputs['labels'];yte=test_outputs['labels']
    if len(np.unique(ytr))<2 or not set(np.unique(yte)).issubset(set(ytr)):
        raise ValueError('Linear probe requires >=2 deleted classes represented in both partitions.')
    model=make_pipeline(StandardScaler(),RidgeClassifier(alpha=1.0))
    model.fit(train_outputs['features'],ytr)
    return 100*accuracy_score(yte,model.predict(test_outputs['features']))

def attack_features(out):
    p=out['probs'].astype('float64');y=out['labels']
    p=np.clip(p,1e-12,1.)
    top=np.sort(p,axis=1)[:,-min(3,p.shape[1]):]
    top=np.pad(top,((0,0),(max(0,3-top.shape[1]),0)),constant_values=0)
    highest=top[:,-1];second=top[:,-2]
    truth=p[np.arange(len(y)),y]
    ent=-np.sum(p*np.log(p),axis=1)
    return np.column_stack([top,highest-second,ent,truth,-np.log(truth)])

def own_attack_auc(train_outputs,test_outputs):
    xm=attack_features(train_outputs)
    xn=attack_features(test_outputs)
    features=np.concatenate([xm,xn],axis=0)
    labels=np.r_[np.ones(len(xm),dtype=int),np.zeros(len(xn),dtype=int)]
    idx=np.arange(len(labels))
    itr,ite=train_test_split(idx,test_size=MIA_ATTACK_TEST_FRACTION,
                             random_state=TRAIN_SEED,stratify=labels)
    classifier=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,
                                         class_weight='balanced',random_state=TRAIN_SEED))
    classifier.fit(features[itr],labels[itr])
    scores=classifier.predict_proba(features[ite])[:,1]
    return float(roc_auc_score(labels[ite],scores))

print('Metric functions ready: JSD, entropy, 3 probes and 2 own-MIA AUCs.')

Metric functions ready: JSD, entropy, 3 probes and 2 own-MIA AUCs.


## 7 · Run **only two** configurations, with a fresh four-hour deadline

The timer begins now (not during downloading). The runner saves a completed metrics row immediately after it finishes and stops between epochs safely when the internal budget is exhausted. **Kaggle's session can still expire earlier.**

In [9]:
SUMMARY_FILE=OUTPUT_DIR/'deleted_class_missing_metrics_12.csv'  # same schema as original 12-row run
ERRORS_FILE=OUTPUT_DIR/'errors.csv'

# Save a fresh, user-downloadable backup immediately after each completed scope.
def save_progress_backup():
    archive_path = Path('/kaggle/working/stanford_dogs_10_15_backup.zip')
    temp = archive_path.with_name(archive_path.name+'.tmp')
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for filename in ('dataset_manifest.json','deleted_class_missing_metrics_12.csv','errors.csv'):
            p=OUTPUT_DIR/filename
            if p.is_file(): z.write(p,arcname='deleted_class_diagnostics/'+filename)
        for folder in ('checkpoints','individual_metrics'):
            base=OUTPUT_DIR/folder
            if base.is_dir():
                for p in base.rglob('*'):
                    if p.is_file():
                        z.write(p,arcname='deleted_class_diagnostics/'+str(p.relative_to(OUTPUT_DIR)))
    os.replace(temp,archive_path)
    print('BACKUP SAVED:',archive_path,flush=True)


def experiment_id(name,scope):
    inp={'dataset':name,'scope':scope,'manifest':DATASET_MANIFEST[name]['fingerprint'],
         'classes':[RAW_DATA[name]['classes'][i] for i in FORGET_CLASSES[(name,scope)]],
         'models':{r:model_metadata(name,0 if r=='teacher' else scope,r)['experiment_id']
                   for r in ('teacher','student','reference')},
         'mia_pairs_per_class':MIA_PAIRS_PER_RETAINED_CLASS,
         'mia_split':MIA_ATTACK_TEST_FRACTION,
         'algorithm':'diagnostics_v2'}
    return hashlib.sha256(json.dumps(inp,sort_keys=True).encode()).hexdigest()[:20]

def evaluate_one(name,scope):
    selection=select_indices(name,scope)
    miam, mian=mia_pairs(name,scope)
    if any(len(selection[x])==0 for x in ('forget_train','forget_test','retain_train','retain_test')):
        raise ValueError('Selected forget/retain train/test partition empty')
    models_loaded={};metas={};sources={};provenance={}
    for role in ('teacher','student','reference'):
        use_scope=0 if role=='teacher' else scope
        mod,meta,path,origin=output_checkpoint(name,use_scope,role)
        models_loaded[role]=mod
        metas[role]=meta;sources[role]=str(path);provenance[role]=origin
        print('MODEL',role,origin,path,flush=True)

    if DEADLINE is not None and time.monotonic()>=DEADLINE:
        raise TimeBudgetPause('All model checkpoints are saved; evaluation will resume next session.')
    role_outputs={}
    try:
        for role,model in models_loaded.items():
            use_scope=0 if role=='teacher' else scope
            meta=metas[role]
            role_outputs[role]={
                'forget_train':cached_outputs(name,use_scope,role,model,meta,'forget_train',selection['forget_train']),
                'forget_test':cached_outputs(name,use_scope,role,model,meta,'forget_test',selection['forget_test']),
            }
            if role!='teacher':
                role_outputs[role]['mia_train']=cached_outputs(name,use_scope,role,model,meta,'mia_train',miam)
                role_outputs[role]['mia_test']=cached_outputs(name,use_scope,role,model,meta,'mia_test',mian)
    finally:
        for model in models_loaded.values(): del model
        models_loaded.clear()
        torch.cuda.empty_cache()

    s=role_outputs['student'];r=role_outputs['reference'];t=role_outputs['teacher']
    if not np.array_equal(s['forget_test']['labels'],r['forget_test']['labels']):
        raise AssertionError('Student and reference forgotten-test order is not identical')
    row={
        'Dataset':name,'Forget scope':f'{scope} classes',
        'JS divergence (nats)':round(mean_jsd(s['forget_test']['probs'],r['forget_test']['probs']),6),
        'Retrained entropy (nats)':round(entropy_nats(r['forget_test']['probs']),6),
        'Teacher linear probe (%)':round(probe_accuracy(t['forget_train'],t['forget_test']),4),
        'Retrained linear probe (%)':round(probe_accuracy(r['forget_train'],r['forget_test']),4),
        'Student linear probe (%)':round(probe_accuracy(s['forget_train'],s['forget_test']),4),
        'Student-own MIA AUC (retain)':round(own_attack_auc(s['mia_train'],s['mia_test']),6),
        'Retrained-own MIA AUC (retain)':round(own_attack_auc(r['mia_train'],r['mia_test']),6),
        'N forgotten train':len(selection['forget_train']),
        'N forgotten test':len(selection['forget_test']),
        'N attack members':len(miam),'N attack nonmembers':len(mian),
        'Experiment ID':experiment_id(name,scope),
        'Dataset mode':DATA_MODE,'Dataset fingerprint':DATASET_MANIFEST[name]['fingerprint'],
        'Teacher provenance':provenance['teacher'],
        'Student provenance':provenance['student'],
        'Reference provenance':provenance['reference'],
        'Paper matched':bool(DATA_MODE=='original_balanced' and all(x=='verified_original' for x in provenance.values())),
        'Forgotten classes':json.dumps([RAW_DATA[name]['classes'][i] for i in FORGET_CLASSES[(name,scope)]]),
        'Checkpoint paths':json.dumps(sources),
    }
    detail_dir=OUTPUT_DIR/'individual_metrics';detail_dir.mkdir(exist_ok=True)
    path=detail_dir/(model_slug(name,scope,'metrics')+'_'+row['Experiment ID']+'.json')
    path.write_text(json.dumps(row,indent=2))
    return row

def run_all_metrics():
    rows=(pd.read_csv(SUMMARY_FILE).to_dict('records') if SUMMARY_FILE.exists() else [])
    completed={(r.get('Dataset'),r.get('Forget scope'),str(r.get('Experiment ID')))
               for r in rows}
    completed_new=0
    for name in RUN_DATASETS:
        for scope in FORGET_SCOPES:
            key=(name,f'{scope} classes',experiment_id(name,scope))
            if key in completed:
                print('SKIP saved configuration',name,scope,flush=True);continue
            if DEADLINE is not None and time.monotonic()>=DEADLINE:
                print('TIME-BUDGET PAUSE: no further experiments started.');return pd.DataFrame(rows)
            if MAX_NEW_EXPERIMENTS_PER_SESSION is not None and completed_new>=MAX_NEW_EXPERIMENTS_PER_SESSION:
                print('CONFIGURATION LIMIT: save results and resume next session.');return pd.DataFrame(rows)
            print('\n'+'='*68+'\nEVALUATE',name,'Forget',scope,flush=True)
            try:
                result=evaluate_one(name,scope)
                rows=[r for r in rows if not (r.get('Dataset')==name and r.get('Forget scope')==f'{scope} classes')]
                rows.append(result)
                pd.DataFrame(rows).to_csv(SUMMARY_FILE,index=False)
                completed_new+=1
                print('SAVED completed metrics',name,scope,'|',SUMMARY_FILE,flush=True)
                save_progress_backup()
                display(pd.DataFrame([result]).iloc[:,:9])
            except TimeBudgetPause as exc:
                print('SAFE PAUSE:',exc,flush=True)
                save_progress_backup()
                return pd.DataFrame(rows)
            except Exception as exc:
                failed={'Dataset':name,'Forget scope':scope,'Error':repr(exc),'When':time.time()}
                old=pd.read_csv(ERRORS_FILE) if ERRORS_FILE.exists() else pd.DataFrame()
                pd.concat([old,pd.DataFrame([failed])],ignore_index=True).to_csv(ERRORS_FILE,index=False)
                raise
    return pd.DataFrame(rows)

assert RUN_DATASETS == ['Stanford Dogs'] and FORGET_SCOPES == [10, 15]
START_MONOTONIC = time.monotonic()
DEADLINE = START_MONOTONIC + TIME_BUDGET_MINUTES * 60
print(f'START 4-HOUR INTERNAL WINDOW: {TIME_BUDGET_MINUTES} minutes, only Stanford Dogs 10 & 15.')
results=run_all_metrics()
print('\nCompleted rows for Stanford Dogs:',len(results),'of',2)
print('CSV:',SUMMARY_FILE)

START 4-HOUR INTERNAL WINDOW: 240 minutes, only Stanford Dogs 10 & 15.

EVALUATE Stanford Dogs Forget 10
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]


BEGIN Stanford Dogs | teacher | Forget-0 | N-train=16469 | epochs 1-30
  epoch  1/30: loss=2.2397, time=62.1s
  epoch  2/30: loss=1.0456, time=40.9s
  epoch  3/30: loss=0.6715, time=40.7s
  epoch  4/30: loss=0.4425, time=40.7s
  epoch  5/30: loss=0.3072, time=40.9s
  epoch  6/30: loss=0.2213, time=40.9s
  epoch  7/30: loss=0.1702, time=40.8s
  epoch  8/30: loss=0.1446, time=41.0s
  epoch  9/30: loss=0.1189, time=40.6s
  epoch 10/30: loss=0.1088, time=40.5s
  epoch 11/30: loss=0.0975, time=41.0s
  epoch 12/30: loss=0.0970, time=40.9s
  epoch 13/30: loss=0.0843, time=41.0s
  epoch 14/30: loss=0.0969, time=40.8s
  epoch 15/30: loss=0.0844, time=41.0s
  epoch 16/30: loss=0.0750, time=41.0s
  epoch 17/30: loss=0.0769, time=40.8s
  epoch 18/30: loss=0.0637, time=40.9s
  epoch 19/30: loss=0.0624, time=40.7s
  epoch 20/30: loss=0.0731, time=41.1s
  epoch 21/30: loss=0.0665, time=41.4s
  epoch 22/30: loss=0.0646, time=41.5s
  epoch 23/30: loss=0.0605, time=41.4s
  epoch 24/30: loss=0.0585, time

,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,Stanford Dogs,10 classes,0.481282,0.871692,92.4765,90.9091,73.9812,0.849856,0.850474



EVALUATE Stanford Dogs Forget 15
MODEL teacher reconstructed /kaggle/working/deleted_class_diagnostics/checkpoints/stanford_dogs_forget0_teacher_852314d5262dbbeff2ef.pt
BEGIN Stanford Dogs | student | Forget-15 | N-train=14498 | epochs 1-30
  epoch  1/30: loss=22.4938, time=47.5s
  epoch  2/30: loss=20.6213, time=37.2s
  epoch  3/30: loss=19.3698, time=37.3s
  epoch  4/30: loss=18.1658, time=37.6s
  epoch  5/30: loss=16.9248, time=37.3s
  epoch  6/30: loss=15.7007, time=37.9s
  epoch  7/30: loss=14.4976, time=37.2s
  epoch  8/30: loss=13.4096, time=37.3s
  epoch  9/30: loss=12.3852, time=37.4s
  epoch 10/30: loss=11.5183, time=37.4s
  epoch 11/30: loss=10.6598, time=37.3s
  epoch 12/30: loss=9.7721, time=37.6s
  epoch 13/30: loss=9.0325, time=37.3s
  epoch 14/30: loss=8.2551, time=37.3s
  epoch 15/30: loss=7.5179, time=37.0s
  epoch 16/30: loss=6.7384, time=37.3s
  epoch 17/30: loss=6.1148, time=37.3s
  epoch 18/30: loss=5.4882, time=37.3s
  epoch 19/30: loss=4.9914, time=37.6s
  epoc

,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,Stanford Dogs,15 classes,0.454464,0.850701,90.4472,86.5854,73.9837,0.842698,0.835356



Completed rows for Stanford Dogs: 2 of 2
CSV: /kaggle/working/deleted_class_diagnostics/deleted_class_missing_metrics_12.csv


## 8 · Export the two missing rows (CSV, LaTeX and one-click checkpoint backup)

Blank cells mean training/evaluation has not completed. The seven metrics are never estimated. Exported table includes **only Stanford Dogs Forget-10 and Forget-15**, so you can paste the results into the two remaining rows of your earlier 10/12 diagnostic table.

In [12]:
METRICS=['JS divergence (nats)','Retrained entropy (nats)',
         'Teacher linear probe (%)','Retrained linear probe (%)','Student linear probe (%)',
         'Student-own MIA AUC (retain)','Retrained-own MIA AUC (retain)']
all_rows=pd.DataFrame([{'Dataset':name,'Forget scope':f'{s} classes'}
                       for name in RUN_DATASETS for s in FORGET_SCOPES])
if SUMMARY_FILE.exists():
    df=pd.read_csv(SUMMARY_FILE)
    # Keep ONLY rows with the current data+model identities.
    expected={(name,f'{s} classes'):experiment_id(name,s)
              for name in RUN_DATASETS for s in FORGET_SCOPES}
    df=df[df.apply(lambda r:str(r.get('Experiment ID'))==expected.get((r['Dataset'],r['Forget scope']),''),axis=1)]
    df=df.drop_duplicates(['Dataset','Forget scope'],keep='last')
    table=all_rows.merge(df[['Dataset','Forget scope']+METRICS],on=['Dataset','Forget scope'],how='left')
else:
    table=all_rows.copy()
    for col in METRICS:table[col]=np.nan

print('COMPLETED:',table[METRICS[0]].notna().sum(),'/ 2')
display(table.style.format({col:'{:.4f}' for col in METRICS},na_rep='—'))
output=OUTPUT_DIR/'stanford_dogs_forget10_15_metrics.csv'
table.to_csv(output,index=False)
valid_provenance=(SUMMARY_FILE.exists() and len(pd.read_csv(SUMMARY_FILE))==2
                  and pd.read_csv(SUMMARY_FILE)['Paper matched'].astype(str).str.lower().eq('true').all())
if not valid_provenance:
    print('WARNING: One or more results use recreated/unverified weights or missing data. '
          'Do NOT describe this as a verified historical-model comparison.')
latex=table.to_latex(index=False,escape=True,na_rep='---',float_format='%.4f',
  caption='Stanford Dogs Forget-10 and Forget-15 deleted-class diagnostics (reconstructed or verified provenance '
          'must be documented separately). Own-MIA AUC uses retained train/test membership.',
  label='tab:deleted_class_retrained_diagnostics')
(OUTPUT_DIR/'stanford_dogs_forget10_15_metrics.tex').write_text(latex)
print('Saved:',output)
print('Saved:',OUTPUT_DIR/'stanford_dogs_forget10_15_metrics.tex')
if table[METRICS[0]].notna().sum()==2:
    print('Both Stanford Dogs rows computed. Check dataset mode and model provenance before any paper claim.')
else:
    print('Partial table only: resume remaining model training/evaluations from checkpoints.')

# Backup essential outputs, including progress checkpoints, WITHOUT huge inference caches.
import zipfile
archive_path = Path('/kaggle/working/stanford_dogs_10_15_backup.zip')
include_dirs = ['checkpoints', 'individual_metrics']
include_files = ['dataset_manifest.json', 'deleted_class_missing_metrics_12.csv',
                 'stanford_dogs_forget10_15_metrics.csv', 'stanford_dogs_forget10_15_metrics.tex',
                 'errors.csv']
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for file_name in include_files:
        path = OUTPUT_DIR / file_name
        if path.is_file():
            z.write(path, arcname='deleted_class_diagnostics/' + path.name)
    for folder_name in include_dirs:
        root = OUTPUT_DIR / folder_name
        if root.is_dir():
            for path in root.rglob('*'):
                if path.is_file():
                    z.write(path, arcname='deleted_class_diagnostics/' + str(path.relative_to(OUTPUT_DIR)))
print('BACKUP READY:', archive_path, 'size:', round(archive_path.stat().st_size/1024**2, 1), 'MiB')
print('Download this ZIP from Kaggle Outputs or Save Version so progress survives a new session.')

COMPLETED: 2 / 2


,Dataset,Forget scope,JS divergence (nats),Retrained entropy (nats),Teacher linear probe (%),Retrained linear probe (%),Student linear probe (%),Student-own MIA AUC (retain),Retrained-own MIA AUC (retain)
0,Stanford Dogs,10 classes,0.4813,0.8717,92.4765,90.9091,73.9812,0.8499,0.8505
1,Stanford Dogs,15 classes,0.4545,0.8507,90.4472,86.5854,73.9837,0.8427,0.8354


Saved: /kaggle/working/deleted_class_diagnostics/stanford_dogs_forget10_15_metrics.csv
Saved: /kaggle/working/deleted_class_diagnostics/stanford_dogs_forget10_15_metrics.tex
Both Stanford Dogs rows computed. Check dataset mode and model provenance before any paper claim.
BACKUP READY: /kaggle/working/stanford_dogs_10_15_backup.zip size: 199.7 MiB
Download this ZIP from Kaggle Outputs or Save Version so progress survives a new session.


## 9 · How to reuse the results and preserve progress

- Final Stanford Dog rows are in `/kaggle/working/deleted_class_diagnostics/stanford_dogs_forget10_15_metrics.csv` and the matching `.tex` file.
- Detailed model provenance and per-experiment JSON are in `deleted_class_diagnostics/individual_metrics/`.
- A compact backup containing model/progress checkpoints and results is at `/kaggle/working/stanford_dogs_10_15_backup.zip`. Download it **before the session expires** or save a Kaggle notebook version with outputs, and verify the files are present.
- To resume a future session, restore the `deleted_class_diagnostics` folder from the ZIP or attach prior notebook saved output and set `PREVIOUS_RESULTS_DIR` (before the first training cell). With missing checkpoints, the models must retrain.
- These public-demo diagnostics are measured values for **reconstructed models and a public data split**, not a verified repeat of your paper's historical balanced dataset experiments. Do not claim otherwise in the manuscript.